# Asynchonous scraping

In [1]:
!pip install aiohttp
!pip install nest-asyncio

In [10]:
import asyncio
import requests
from bs4 import BeautifulSoup
import csv
import re

In [11]:
# ?? lock para evitar conflito ao escrever no arquivo
lock = asyncio.Lock()


In [12]:
def extract_links(text):
    soup = BeautifulSoup(text, 'html.parser')
    return [
        link.get('href')
        for link in soup.find_all('a', href=re.compile('^http'))
    ]

In [13]:
def fetch_sync(url):
    try:
        response = requests.get(url, timeout=10, headers={
            "User-Agent": "Mozilla/5.0"
        })
        return response.text
    except Exception as e:
        print(f"Erro em {url}: {e}")
        return None

In [19]:
async def save_links(links):
    async with lock:
        with open('csv_file.csv', 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            for link in links:
                writer.writerow([link])

In [15]:
async def process_url(url):
    # roda requests em thread separada
    text = await asyncio.to_thread(fetch_sync, url)

    if text:
        links = extract_links(text)
        await save_links(links)

In [16]:
async def scrap(urls):
    tasks = [process_url(url) for url in urls]
    await asyncio.gather(*tasks)


In [20]:
# ? URLs
urls = [
    'https://analytics.usa.gov/',
    'https://www.python.org/',
    'https://www.linkedin.com/'
]

await scrap(urls)